# Feature Engineering

Gera dois artefatos:

| Arquivo | Chave | Conteúdo |
|---|---|---|
| `features/ds_features.parquet` | `unique_id × ds` | Calendar sin/cos, product attrs, entity IDs |
| `features/ds_holidays.parquet` | `ds × region_id` | Feriados nacionais + regionais + proximidade |

**Features estáticas** (`ds_features`): constantes por `unique_id`, válidas para qualquer semana.
**Features de feriado** (`ds_holidays`): dinâmicas por `ds`, variam por macro-região (5 regiões → estados brasileiros).

Stats (cv, iqr, q50) são excluídas — computadas no model_building após o split para evitar leakage.

In [1]:
import os
import unicodedata
import pandas as pd
import numpy as np
import warnings
from holidays import country_holidays
warnings.filterwarnings('ignore')

In [2]:
# Inputs
PATH_REFINED   = '../../data/gold/forecasting/datasets/refined'
PATH_BASE      = '../../data/gold/forecasting/datasets/base/ds_sales_timeseries.parquet'
PATH_DIM_REGION = '../../data/gold/data_warehouse/dw_dim_region.parquet'

# Outputs
PATH_FEATURES  = '../../data/gold/forecasting/features/ds_features.parquet'
PATH_HOLIDAYS  = '../../data/gold/forecasting/features/ds_holidays.parquet'
os.makedirs(os.path.dirname(PATH_FEATURES), exist_ok=True)

DEMAND_TYPES = ['smooth', 'erratic', 'intermittent', 'lumpy']

## 1. Carrega séries válidas (refined)

In [3]:
df_valid = pd.concat([
    pd.read_parquet(f'{PATH_REFINED}/ds_sales_timeseries_{dt}.parquet')
    for dt in DEMAND_TYPES
], ignore_index=True)

print(f'Séries válidas : {df_valid["unique_id"].nunique():,}')
print(f'Linhas totais  : {len(df_valid):,}')
print(f'Colunas        : {df_valid.columns.tolist()}')

Séries válidas : 2,232
Linhas totais  : 186,052
Colunas        : ['unique_id', 'ds', 'y', 'demand_type']


## 2. Puxar atributos do base dataset

In [4]:
# Colunas necessárias do base dataset (catálogo de atributos)
BASE_COLS = [
    'time_series_id', 'week_date',
    # Calendar
    'week', 'month', 'quarter', 'year',
    # Product
    'product_attribute_1', 'product_attribute_2', 'product_attribute_3',
    # Entity IDs
    'supplier_id', 'region_id',
]

df_base = pd.read_parquet(PATH_BASE, columns=BASE_COLS)

print(f'Base dataset carregado: {df_base.shape}')
print('Nota: stats (cv, iqr, q50) são computadas no model_building após o split — evita leakage.')

Base dataset carregado: (203471, 11)
Nota: stats (cv, iqr, q50) são computadas no model_building após o split — evita leakage.


## 3. Build features

In [5]:
def build_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Encoding cíclico — captura proximidade entre mês 12 e mês 1, etc."""
    df['month_sin']   = np.sin(2 * np.pi * df['month']   / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month']   / 12)
    df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
    df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)
    df['week_sin']    = np.sin(2 * np.pi * df['week']    / 52)
    df['week_cos']    = np.cos(2 * np.pi * df['week']    / 52)
    return df.drop(columns=['week', 'month', 'quarter'])  # mantém year como numérico


def build_product_features(df: pd.DataFrame) -> pd.DataFrame:
    """Label encoding dos atributos categóricos de produto."""
    for col in ['product_attribute_1', 'product_attribute_2', 'product_attribute_3']:
        df[col] = df[col].astype('category').cat.codes
    return df


def build_features(
    df_valid: pd.DataFrame,
    df_base: pd.DataFrame,
) -> pd.DataFrame:
    """
    Gera features estáticas sem leakage:
      - Calendar sin/cos (determinístico)
      - Product attributes (metadado estático)
      - supplier_id, region_id (identificadores estáticos)

    Stats (cv, iqr, q50) são excluídas — devem ser computadas
    no model_building após o train/test split para evitar leakage.
    """
    df = df_valid[['unique_id', 'ds']].merge(
        df_base.rename(columns={'time_series_id': 'unique_id', 'week_date': 'ds'}),
        on=['unique_id', 'ds'],
        how='left'
    )
    df = build_calendar_features(df)
    df = build_product_features(df)

    return df.sort_values(['unique_id', 'ds']).reset_index(drop=True)

In [6]:
df_features = build_features(df_valid, df_base)

print(f'Shape    : {df_features.shape}')
print(f'Colunas  : {df_features.columns.tolist()}')
print(f'NaN (%)  :\n{df_features.isna().mean().sort_values(ascending=False).head(5)}')
df_features.head()

Shape    : (186052, 14)
Colunas  : ['unique_id', 'ds', 'year', 'product_attribute_1', 'product_attribute_2', 'product_attribute_3', 'supplier_id', 'region_id', 'month_sin', 'month_cos', 'quarter_sin', 'quarter_cos', 'week_sin', 'week_cos']
NaN (%)  :
unique_id              0.0
ds                     0.0
year                   0.0
product_attribute_1    0.0
product_attribute_2    0.0
dtype: float64


,unique_id,ds,year,product_attribute_1,product_attribute_2,product_attribute_3,supplier_id,region_id,month_sin,month_cos,quarter_sin,quarter_cos,week_sin,week_cos
0,2,2023-03-06,2023,0,0,0,1,5,1.000000,6.123234e-17,1.000000e+00,6.123234e-17,0.935016,3.546049e-01
1,2,2023-03-13,2023,0,0,0,1,5,1.000000,6.123234e-17,1.000000e+00,6.123234e-17,0.970942,2.393157e-01
2,2,2023-03-20,2023,0,0,0,1,5,1.000000,6.123234e-17,1.000000e+00,6.123234e-17,0.992709,1.205367e-01
3,2,2023-03-27,2023,0,0,0,1,5,1.000000,6.123234e-17,1.000000e+00,6.123234e-17,1.000000,-1.608123e-16
4,2,2023-04-03,2023,0,0,0,1,5,0.866025,-5.000000e-01,1.224647e-16,-1.000000e+00,0.992709,-1.205367e-01


## 4. Salvar — Features estáticas

In [ ]:
df_features.to_parquet(PATH_FEATURES, index=False)

print(f'Salvo em : {PATH_FEATURES}')
print(f'Shape    : {df_features.shape}')
print()
print('Colunas por grupo:')
cal_cols  = [c for c in df_features.columns if any(s in c for s in ['sin','cos','year'])]
stat_cols = ['cv', 'iqr', 'q50_sales']
prod_cols = [c for c in df_features.columns if 'attribute' in c]
id_cols   = ['supplier_id', 'region_id']
print(f'  Calendar : {cal_cols}')
print(f'  Static   : {stat_cols}')
print(f'  Product  : {prod_cols}')
print(f'  IDs      : {id_cols}')

## 5. Holiday Features

Features de feriado keyed por `(ds, region_id)`.

**Macro-região → estados** (para feriados estaduais):

| region_id | Região | Estados |
|---|---|---|
| 1 | Centro-Oeste | DF, GO, MS, MT |
| 2 | Nordeste | AL, BA, CE, MA, PB, PE, PI, RN, SE |
| 3 | Norte | AC, AM, AP, PA, RO, RR, TO |
| 4 | Sudeste | ES, MG, RJ, SP |
| 5 | Sul | PR, RS, SC |

**Colunas geradas:**
- `is_national_holiday` — feriado nacional na semana
- `is_regional_holiday` — feriado estadual exclusivo da região na semana
- `is_holiday` — qualquer feriado (nacional ou regional)
- `n_holidays` — total de feriados na semana
- `holiday_type` — tipo principal: `none / regional / other / christmas_newyear / easter / carnival`
- `holiday_type_enc` — encoding ordinal por impacto esperado (0=none … 5=carnival)
- `weeks_to_next_holiday` — semanas até o próximo feriado, cap 4
- `weeks_since_last_holiday` — semanas desde o último feriado, cap 4

In [4]:
# Macro-região → estados brasileiros
REGION_TO_STATES = {
    1: ['DF', 'GO', 'MS', 'MT'],
    2: ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'],
    3: ['AC', 'AM', 'AP', 'PA', 'RO', 'RR', 'TO'],
    4: ['ES', 'MG', 'RJ', 'SP'],
    5: ['PR', 'RS', 'SC'],
}

# Prioridade por impacto na demanda farmacêutica
HOLIDAY_TYPE_KEYWORDS = [
    ('carnival',          ['carnaval', 'quaresma']),
    ('easter',            ['sexta-feira santa', 'corpus christi']),
    ('christmas_newyear', ['natal', 'ano-novo', 'confraterniza']),
    ('other',             []),   # fallback para demais nacionais
]
HOLIDAY_TYPE_ENC = {
    'none': 0, 'regional': 1, 'other': 2,
    'christmas_newyear': 3, 'easter': 4, 'carnival': 5,
}


def _strip_accents(text: str) -> str:
    nfkd = unicodedata.normalize('NFD', text.lower())
    return ''.join(c for c in nfkd if unicodedata.category(c) != 'Mn')


def classify_holiday_type(names: list) -> str:
    combined = _strip_accents(' '.join(names))
    for htype, keywords in HOLIDAY_TYPE_KEYWORDS:
        if any(k in combined for k in keywords):
            return htype
    return 'other'


def _holidays_in_week(week_start: pd.Timestamp, hol_dict: dict) -> dict:
    """Retorna sub-dict com feriados que caem em [week_start, week_start+6d]."""
    ws, we = week_start.date(), (week_start + pd.Timedelta(days=6)).date()
    return {d: n for d, n in hol_dict.items() if ws <= d <= we}


def build_holiday_features(all_weeks: pd.Series) -> pd.DataFrame:
    """
    Gera features de feriado por (ds, region_id).

    Usa categories='public'+'optional' para incluir Carnaval, Corpus Christi
    e vésperas. Feriados estaduais são imputados por macro-região via
    REGION_TO_STATES.
    """
    years = list(range(all_weeks.dt.year.min() - 1, all_weeks.dt.year.max() + 2))

    # Feriados nacionais (valem para todos)
    nat_hols  = country_holidays('BR', years=years, categories=('public', 'optional'))
    nat_dates = set(nat_hols.keys())

    rows = []

    for region_id, states in REGION_TO_STATES.items():
        # Feriados estaduais que NÃO são nacionais
        reg_hols = {}
        for state in states:
            sh = country_holidays('BR', subdiv=state, years=years,
                                  categories=('public', 'optional'))
            for d, n in sh.items():
                if d not in nat_dates:
                    reg_hols[d] = n   # última escrita vence (todos os estados da região)

        for week_start in all_weeks:
            nat_week = _holidays_in_week(week_start, nat_hols)
            reg_week = _holidays_in_week(week_start, reg_hols)

            n_nat  = len(nat_week)
            n_reg  = len(reg_week)
            n_tot  = n_nat + n_reg
            is_any = int(n_tot > 0)

            if n_nat > 0:
                htype = classify_holiday_type(list(nat_week.values()))
            elif n_reg > 0:
                htype = 'regional'
            else:
                htype = 'none'

            rows.append({
                'ds':                  week_start,
                'region_id':           region_id,
                'is_national_holiday': int(n_nat > 0),
                'is_regional_holiday': int(n_reg > 0),
                'is_holiday':          is_any,
                'n_holidays':          n_tot,
                'holiday_type':        htype,
                'holiday_type_enc':    HOLIDAY_TYPE_ENC[htype],
            })

    df = pd.DataFrame(rows)

    # Proximidade: semanas até o próximo / desde o último feriado (cap=4)
    def add_proximity(grp: pd.DataFrame) -> pd.DataFrame:
        grp = grp.sort_values('ds').copy()
        flags = grp['is_holiday'].values
        n = len(flags)
        to_next    = np.full(n, 4, dtype=np.int8)
        since_last = np.full(n, 4, dtype=np.int8)
        for i in range(n):
            if flags[i]:
                to_next[i] = since_last[i] = 0
                continue
            for j in range(1, 5):
                if i + j < n and flags[i + j]:
                    to_next[i] = j
                    break
            for j in range(1, 5):
                if i - j >= 0 and flags[i - j]:
                    since_last[i] = j
                    break
        grp['weeks_to_next_holiday']    = to_next
        grp['weeks_since_last_holiday'] = since_last
        return grp

    df = df.groupby('region_id', group_keys=False).apply(add_proximity)
    return df.sort_values(['region_id', 'ds']).reset_index(drop=True)

In [7]:
all_weeks = df_valid['ds'].drop_duplicates().sort_values().reset_index(drop=True)

df_holidays = build_holiday_features(all_weeks)

print(f'Shape    : {df_holidays.shape}')
print(f'Colunas  : {df_holidays.columns.tolist()}')
print(f'Regioes  : {df_holidays["region_id"].unique()}')
print(f'Semanas  : {df_holidays["ds"].nunique()}')
print()
print('Feriados por tipo:')
print(df_holidays[df_holidays['region_id'] == 4]  # Sudeste como exemplo
      .groupby('holiday_type')['is_holiday'].sum()
      .sort_values(ascending=False))
print()
print('Exemplo — Carnaval 2023 (Sudeste):')
mask = (df_holidays['region_id'] == 4) & (df_holidays['ds'].dt.month == 2) & (df_holidays['ds'].dt.year == 2023)
print(df_holidays[mask][['ds','is_holiday','n_holidays','holiday_type','weeks_to_next_holiday','weeks_since_last_holiday']].to_string(index=False))
df_holidays.head(10)

Shape    : (520, 10)
Colunas  : ['ds', 'region_id', 'is_national_holiday', 'is_regional_holiday', 'is_holiday', 'n_holidays', 'holiday_type', 'holiday_type_enc', 'weeks_to_next_holiday', 'weeks_since_last_holiday']
Regioes  : [1 2 3 4 5]
Semanas  : 104

Feriados por tipo:
holiday_type
other                13
christmas_newyear     5
regional              5
easter                4
carnival              2
none                  0
Name: is_holiday, dtype: int64

Exemplo — Carnaval 2023 (Sudeste):
        ds  is_holiday  n_holidays holiday_type  weeks_to_next_holiday  weeks_since_last_holiday
2023-02-06           0           0         none                      2                         4
2023-02-13           0           0         none                      1                         4
2023-02-20           1           3     carnival                      0                         0
2023-02-27           0           0         none                      4                         1


,ds,region_id,is_national_holiday,is_regional_holiday,is_holiday,n_holidays,holiday_type,holiday_type_enc,weeks_to_next_holiday,weeks_since_last_holiday
0,2022-10-31,1,1,0,1,1,other,2,0,0
1,2022-11-07,1,0,0,0,0,none,0,1,1
2,2022-11-14,1,1,1,1,2,other,2,0,0
3,2022-11-21,1,0,0,0,0,none,0,1,1
4,2022-11-28,1,0,1,1,1,regional,1,0,0
5,2022-12-05,1,0,0,0,0,none,0,2,1
6,2022-12-12,1,0,0,0,0,none,0,1,2
7,2022-12-19,1,1,0,1,2,christmas_newyear,3,0,0
8,2022-12-26,1,1,0,1,2,christmas_newyear,3,0,0
9,2023-01-02,1,0,0,0,0,none,0,4,1


## 6. Salvar — Holiday Features

In [6]:
df_holidays.to_parquet(PATH_HOLIDAYS, index=False)

print(f'Salvo em : {PATH_HOLIDAYS}')
print(f'Shape    : {df_holidays.shape}')
print()
print('Colunas por grupo:')
print(f'  Chave       : ["ds", "region_id"]')
print(f'  Flags       : ["is_national_holiday", "is_regional_holiday", "is_holiday", "n_holidays"]')
print(f'  Tipo        : ["holiday_type", "holiday_type_enc"]')
print(f'  Proximidade : ["weeks_to_next_holiday", "weeks_since_last_holiday"]')
print()
print('Como usar no model_building:')
print('  df_train = df_train.merge(df_holidays, on=["ds", "region_id"], how="left")')

Salvo em : ../../data/gold/forecasting/features/ds_holidays.parquet
Shape    : (520, 10)

Colunas por grupo:
  Chave       : ["ds", "region_id"]
  Flags       : ["is_national_holiday", "is_regional_holiday", "is_holiday", "n_holidays"]
  Tipo        : ["holiday_type", "holiday_type_enc"]
  Proximidade : ["weeks_to_next_holiday", "weeks_since_last_holiday"]

Como usar no model_building:
  df_train = df_train.merge(df_holidays, on=["ds", "region_id"], how="left")
